# 05 — emailwerk parity

The jaen emailwerk client against the emailwerk GraphQL API: every
operation the client encodes must exist server-side with the same
args wrapper, and the multi-recipient send contract must hold.
Live checks run only when an emailwerk instance answers on
`JAEN_EMAILWERK_URL`.


In [ ]:
import jaen_testkit as k
k.start_run('05-emailwerk')
print(k.CONFIG['repo_root'])

In [ ]:
import os, re

CLIENT_DIR = k.repo_path('packages/gatsby-jaen-emailwerk')
EW = k.CONFIG['emailwerk_dir']

def root_ops(source, root):
    """Field names of generatedSchema.<root> via brace counting."""
    m = re.search(r'\b%s:\s*\{' % root, source)
    if not m:
        return set()
    depth = 0
    ops = set()
    i = m.end() - 1
    start = i
    for i in range(start, len(source)):
        ch = source[i]
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                break
    body = source[start:i]
    # field names at depth 1 of the root object
    depth = 0
    for fm in re.finditer(r'[{},]|([A-Za-z_][A-Za-z0-9_]*)\s*:', body):
        tok = fm.group(0)
        if tok == '{':
            depth += 1
        elif tok == '}':
            depth -= 1
        elif tok == ',':
            continue
        elif fm.group(1) and depth == 1:
            ops.add(fm.group(1))
    return ops - {'__typename'}

with k.section('static conformance'):
    with k.check('client schema operations exist server-side') as c:
        schema_path = os.path.join(CLIENT_DIR, 'src', 'client', 'schema.generated.ts')
        client_src = k.read_text(schema_path)
        if client_src is None:
            c.fail('client schema missing: %s' % schema_path, abort=True)
        server_src = k.read_text(os.path.join(EW, 'src', 'index.ts'), '')
        server_ops = set(re.findall(r'^\s{4}(\w+):', server_src, re.M))
        client_ops = root_ops(client_src, 'query') | root_ops(client_src, 'mutation')
        unknown = sorted(op for op in client_ops
                         if op not in server_ops and not op.startswith('__'))
        c.note('%d client root ops: %s' % (len(client_ops), sorted(client_ops)))
        c.expect_true(len(client_ops) > 0, 'extraction found root operations')
        c.expect_equal(unknown, [], 'all client ops known to emailwerk')
        if unknown:
            c.detail('\n'.join(unknown))

    with k.check('sendTemplateMail accepts recipient lists server-side') as c:
        server_src = k.read_text(os.path.join(EW, 'src', 'index.ts'), '')
        c.expect_contains(server_src, 'to: string[]')
        c.expect_contains(server_src, 'normalizeRecipients')

    with k.check('send mutations accept the emailwerk:send role') as c:
        server_src = k.read_text(os.path.join(EW, 'src', 'index.ts'), '')
        c.expect_contains(server_src, '"emailwerk:admin", "emailwerk:send"')


In [ ]:
with k.section('emailwerk own test suite'):
    with k.check('recipient normalization unit tests pass') as c:
        r = k.sh('npx vitest run src/send/recipients.test.ts', cwd=EW, timeout=300)
        if r.rc == 127:
            c.skip('npx/vitest unavailable')
        c.require(r, 'vitest recipients')
        c.expect_contains(r.stdout + r.stderr, 'passed')


## Deployed instance

The live checks run against the DEPLOYED emailwerk
(`JAEN_EMAILWERK_URL`, default `https://emailwerk.com/graphql`),
which sits behind an interim HTTP Basic gate. Credentials come from
`JAEN_EMAILWERK_BASIC` (`user:pass`) or, on this machine, from the
ansible-vault (`emailwerk.basic_user`/`basic_pass`).

When production still runs a build predating the multi-recipient
change, the `to` list check records WARN. To deploy the new build,
run (human step, never from this notebook):

```bash
cd ~/git/emailwerk-fido-test/deploy/k8s/emailwerk && ./redeploy.sh
```


In [ ]:
with k.section('deployed instance'):
    auth = k.emailwerk_auth_headers()
    url = k.CONFIG['emailwerk_url']

    with k.check('anonymous requests are rejected (auth gate up)') as c:
        if not url:
            c.skip('JAEN_EMAILWERK_URL not set')
        r = k.graphql(url, '{ __typename }')
        if r.error:
            c.skip('endpoint not reachable: %s' % k.preview(r.error, 60))
        c.expect_true(r.status in (401, 403), 'HTTP %s without credentials' % r.status)

    with k.check('deployed emailwerk answers introspection') as c:
        if not url:
            c.skip('JAEN_EMAILWERK_URL not set')
        if not auth:
            c.skip('no credentials (JAEN_EMAILWERK_BASIC or vault)')
        r = k.graphql(url, '{ __schema { mutationType { name } } }', headers=auth)
        if r.error:
            c.skip('endpoint not reachable')
        c.expect_true(r.ok, 'HTTP %s' % r.status)
        data = (r.json() or {}).get('data', {}).get('__schema')
        c.expect_true(bool(data), 'schema introspection answered')

    with k.check('deployed schema has the client operations') as c:
        if not url or not auth:
            c.skip('no live credentials')
        r = k.graphql(url, '''
          { q: __type(name: "Query") { fields { name } }
            m: __type(name: "Mutation") { fields { name } } }''', headers=auth)
        if r.error or not r.ok:
            c.skip('no live instance')
        payload = (r.json() or {}).get('data', {})
        live = {f['name'] for t in ('q', 'm') for f in (payload.get(t) or {}).get('fields', [])}
        wanted = {'templates', 'template', 'senders', 'dashboard',
                  'templateCreate', 'templateUpdate', 'templateDelete',
                  'templatePreview', 'senderCreate', 'sendTemplateMail', 'sendEmail'}
        missing = sorted(wanted - live)
        c.note('%d live root fields' % len(live))
        c.expect_equal(missing, [], 'all client operations deployed')

    with k.check('deployed sendTemplateMail accepts recipient lists') as c:
        if not url or not auth:
            c.skip('no live credentials')
        r = k.graphql(url, '''
          { __type(name: "SendTemplateMailArgsInput") {
              inputFields { name type { kind ofType { kind name ofType { name } } } } } }''',
          headers=auth)
        if r.error or not r.ok:
            c.skip('no live instance')
        data = (r.json() or {}).get('data', {}).get('__type')
        if not data:
            c.skip('introspection disabled or type absent')
        to_field = next((f for f in data['inputFields'] if f['name'] == 'to'), None)
        c.expect_true(to_field is not None, 'to field present')
        if to_field and 'LIST' not in str(to_field):
            c.warn('deployed build predates the multi-recipient change '
                   '(to is still a bare String) — redeploy pending')
        elif to_field:
            c.ok('to is list-typed')


In [ ]:
k.summary()
k.save_results('results-05-emailwerk.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'